# Household photo candidates from Dollar Street

This notebook cuts a small bundle out of the Dollar Street dataset that Kaggle already hosts, so nothing large is downloaded to a laptop. No GPU is needed.

1. Create a Kaggle notebook and import this file.
2. Add the input "The Dollar Street Dataset" (MLCommons).
3. Run all cells. It takes a few minutes.
4. Download `household_candidates.zip` (about 30 MB) from the Output tab. The next step, `03_contact_sheets.py`, reads it.

The photos are from Dollar Street (Gapminder and MLCommons), CC BY-SA 4.0. Each photo is resized to 448 pixels on its long side. `found.csv` keeps the Dollar Street id, the country, the home and the topic of every photo. The candidate list in the first code cell comes from `01_select_candidates.py`.

In [ ]:
import os, io, zipfile, collections
import pandas as pd
from PIL import Image
CSV = '''row,group,id,topic,country,place,income
1,west_african,5d4bdf2fcf0b3a0f3f337423,cooking_pots,Togo,kouleta,45.0
1,west_african,5d4bdf33cf0b3a0f3f337473,cooking_pots,Togo,ayanou-ahobli,321.0
1,west_african,5d4bf096cf0b3a0f3f35525e,cooking_pots,Cote d'Ivoire,konan,60.75268981
1,west_african,5d4bf36ecf0b3a0f3f35a020,cooking_pots,Burkina Faso,sinare,44.71243293
1,west_african,5d4bf3c2cf0b3a0f3f35a8be,cooking_pots,Burkina Faso,zongo,55.93351031
1,west_african,5d4bf0b7cf0b3a0f3f355628,cooking_pots,Cote d'Ivoire,yao-bernabi,82.91518231
1,west_african,5d4bf3a5cf0b3a0f3f35a612,cooking_pots,Burkina Faso,bande,52.95427615
1,west_african,5d4bf122cf0b3a0f3f356048,cooking_pots,Cote d'Ivoire,fofana,41.76707666
1,west_african,5d4bdf37cf0b3a0f3f3374eb,cooking_pots,Togo,ayanou-ahobli,321.0
1,west_african,5d4bf470cf0b3a0f3f35ba60,cooking_pots,Burkina Faso,kabore,28.86121301
1,west_african,5d4bdf2acf0b3a0f3f3373a5,cooking_pots,Togo,messan,670.0
1,west_african,5d4bf0dacf0b3a0f3f35599c,cooking_pots,Cote d'Ivoire,ahova,80.72770904
1,west_african,5d4bf0fdcf0b3a0f3f355d1e,cooking_pots,Cote d'Ivoire,coulibaly,68.11250877
1,west_african,5d4bdf25cf0b3a0f3f33730b,cooking_pots,Togo,tchamou,145.0
1,west_african,5d4bf3e1cf0b3a0f3f35ab34,cooking_pots,Burkina Faso,alimata,95.05494965
1,west_african,5d4bf381cf0b3a0f3f35a26a,cooking_pots,Burkina Faso,mwe-quidraogo,53.64520422
1,west_african,5d4bef22cf0b3a0f3f352afc,cooking_pots,Nigeria,ehegwo,59.0
1,west_african,5d4bf05ccf0b3a0f3f354ca0,cooking_pots,Liberia,blamoh,116.0
1,west_african,5d4bf058cf0b3a0f3f354c3a,cooking_pots,Liberia,doe,87.0
1,west_african,5d4bf050cf0b3a0f3f354bbe,cooking_pots,Liberia,galrbah,58.0
1,west_african,5ec4faccf0611d7ddd7422c1,cooking_pots,Ghana,nkrumah,375.0
1,west_african,5d4bef15cf0b3a0f3f35296e,cooking_pots,Nigeria,ibojionu,72.0
1,west_african,5d4bf04ccf0b3a0f3f354b3a,cooking_pots,Liberia,nyanford,48.0
1,west_african,5d4be042cf0b3a0f3f338e76,cooking_pots,Nigeria,ojah,1643.0
1,western,5ec4fc04f0611d7ddd742d06,cooking_pots,France,rayez,875.0
1,western,5d4be545cf0b3a0f3f3419f2,cooking_pots,United States,macintyre,855.0
1,western,5d4be3dacf0b3a0f3f33f0cb,cooking_pots,United States,theresa-s,3450.0
1,western,5d4be1efcf0b3a0f3f33bdb1,cooking_pots,United States,hoffman-tsuhlares,2347.0
1,western,5d4be0decf0b3a0f3f33a05b,cooking_pots,Netherlands,ploum,2370.0
1,western,5ed419e36858701b1019a208,cooking_pots,France,carrillo,7110.0
1,western,5d4bed34cf0b3a0f3f34f2da,cooking_pots,United States,grabowski,2041.0
1,western,5d4be42ecf0b3a0f3f33fa88,cooking_pots,United States,diamond,4446.0
1,western,5ec4f922f0611d7ddd7414c3,cooking_pots,Spain,rodrigo,4256.0
1,western,5ec4f8f7f0611d7ddd74135d,cooking_pots,Spain,garrido,4253.0
1,western,5ec4f5503f62767d97a47319,cooking_pots,France,larriere,19671.0
1,western,5d4bde29cf0b3a0f3f335ad1,cooking_pots,Italy,nazzani,4018.0
2,west_african,5d4bf0b7cf0b3a0f3f355628,cooking,Cote d'Ivoire,yao-bernabi,82.91518231
2,west_african,5d4bf096cf0b3a0f3f35526a,cooking,Cote d'Ivoire,konan,60.75268981
2,west_african,5d4bf0dacf0b3a0f3f3559ae,cooking,Cote d'Ivoire,ahova,80.72770904
2,west_african,5d4bf36ecf0b3a0f3f35a020,cooking,Burkina Faso,sinare,44.71243293
2,west_african,5d4bdf25cf0b3a0f3f33730b,cooking,Togo,tchamou,145.0
2,west_african,5d4bf3c2cf0b3a0f3f35a8be,cooking,Burkina Faso,zongo,55.93351031
2,west_african,5d4bf0b8cf0b3a0f3f355636,cooking,Cote d'Ivoire,yao-bernabi,82.91518231
2,west_african,5d4bf3e1cf0b3a0f3f35ab34,cooking,Burkina Faso,alimata,95.05494965
2,west_african,5d4bf36ecf0b3a0f3f35a02a,cooking,Burkina Faso,sinare,44.71243293
2,west_african,5d4bf122cf0b3a0f3f356048,cooking,Cote d'Ivoire,fofana,41.76707666
2,west_african,5d4bf3e1cf0b3a0f3f35ab3a,cooking,Burkina Faso,alimata,95.05494965
2,west_african,5d4bf3a5cf0b3a0f3f35a612,cooking,Burkina Faso,bande,52.95427615
2,west_african,5d4bf0fdcf0b3a0f3f355d1e,cooking,Cote d'Ivoire,coulibaly,68.11250877
2,west_african,5d4bf3a5cf0b3a0f3f35a61e,cooking,Burkina Faso,bande,52.95427615
2,west_african,5d4bdf33cf0b3a0f3f337473,cooking,Togo,ayanou-ahobli,321.0
2,west_african,5d4bf471cf0b3a0f3f35ba68,cooking,Burkina Faso,kabore,28.86121301
2,west_african,5d4bf0dacf0b3a0f3f35599c,cooking,Cote d'Ivoire,ahova,80.72770904
2,west_african,5d4bf470cf0b3a0f3f35ba60,cooking,Burkina Faso,kabore,28.86121301
2,west_african,5d4bf0fecf0b3a0f3f355d26,cooking,Cote d'Ivoire,coulibaly,68.11250877
2,west_african,5d4bdf2acf0b3a0f3f3373a5,cooking,Togo,messan,670.0
2,west_african,5d4bdf37cf0b3a0f3f3374eb,cooking,Togo,ayanou-ahobli,321.0
2,west_african,5d4bf382cf0b3a0f3f35a272,cooking,Burkina Faso,mwe-quidraogo,53.64520422
2,west_african,5d4bf381cf0b3a0f3f35a26a,cooking,Burkina Faso,mwe-quidraogo,53.64520422
2,west_african,5d4bf096cf0b3a0f3f35525e,cooking,Cote d'Ivoire,konan,60.75268981
2,western,5d4bde28cf0b3a0f3f335aab,stove_hob,Italy,nazzani,4018.0
2,western,605a73b1801a87456085efa1,stove_hob,Denmark,hansen-refsing,5343.0
2,western,5d4be544cf0b3a0f3f3419d2,stove_hob,United States,macintyre,855.0
2,western,5d4be4ebcf0b3a0f3f340f8c,stove_hob,United States,williams,2055.0
2,western,5ec4fc08f0611d7ddd742d20,stove_hob,France,rayez,875.0
2,western,5d4bde8ccf0b3a0f3f3363eb,stove_hob,Netherlands,rasing,3150.0
2,western,5d4be0ddcf0b3a0f3f33a03d,stove_hob,Netherlands,ploum,2370.0
2,western,5d4be3cecf0b3a0f3f33ef6f,stove_hob,United States,theresa-s,3450.0
2,western,5d4bdf9acf0b3a0f3f337ea3,stove_hob,Switzerland,renato,2602.0
2,western,5d4be1d6cf0b3a0f3f33bb19,stove_hob,United States,hoffman-tsuhlares,2347.0
2,western,5ec4f8edf0611d7ddd741306,stove_hob,Spain,garrido,4253.0
2,western,5ec4f8fcf0611d7ddd741382,stove_hob,Spain,dela-torre-sawtana,4553.0
3,west_african,5d4bf07ecf0b3a0f3f35502c,get_water,Cote d'Ivoire,konan,60.75268981
3,west_african,5d4bf35ccf0b3a0f3f359e44,get_water,Burkina Faso,sinare,44.71243293
3,west_african,5d4bf3a4cf0b3a0f3f35a5fa,drinking_water,Burkina Faso,bande,52.95427615
3,west_african,5d4bdf32cf0b3a0f3f33745d,drinking_water,Togo,ayanou-ahobli,321.0
3,west_african,5d4bf3a7cf0b3a0f3f35a666,get_water,Burkina Faso,zongo,55.93351031
3,west_african,5d4bf105cf0b3a0f3f355d64,get_water,Cote d'Ivoire,fofana,41.76707666
3,west_african,5d4bf0b3cf0b3a0f3f3555bc,drinking_water,Cote d'Ivoire,yao-bernabi,82.91518231
3,west_african,5d4bf381cf0b3a0f3f35a258,drinking_water,Burkina Faso,mwe-quidraogo,53.64520422
3,west_african,5d4bf36dcf0b3a0f3f35a000,drinking_water,Burkina Faso,sinare,44.71243293
3,west_african,5d4bf370cf0b3a0f3f35a05e,get_water,Burkina Faso,mwe-quidraogo,53.64520422
3,west_african,5d4bf095cf0b3a0f3f355238,drinking_water,Cote d'Ivoire,konan,60.75268981
3,west_african,5d4bf0fccf0b3a0f3f355d10,drinking_water,Cote d'Ivoire,coulibaly,68.11250877
3,west_african,5d4bf121cf0b3a0f3f35602a,drinking_water,Cote d'Ivoire,fofana,41.76707666
3,west_african,5d4bdf29cf0b3a0f3f337397,drinking_water,Togo,messan,670.0
3,west_african,5d4bf46ecf0b3a0f3f35ba34,drinking_water,Burkina Faso,kabore,28.86121301
3,west_african,5d4bf45bcf0b3a0f3f35b826,get_water,Burkina Faso,kabore,28.86121301
3,west_african,5d4bf3c6cf0b3a0f3f35a930,get_water,Burkina Faso,alimata,95.05494965
3,west_african,5d4bf0e0cf0b3a0f3f355a56,get_water,Cote d'Ivoire,coulibaly,68.11250877
3,west_african,5d4bf0b3cf0b3a0f3f3555b0,drinking_water,Cote d'Ivoire,yao-bernabi,82.91518231
3,west_african,5d4bf0d8cf0b3a0f3f355964,drinking_water,Cote d'Ivoire,ahova,80.72770904
3,west_african,5d4bdf24cf0b3a0f3f337305,drinking_water,Togo,tchamou,145.0
3,west_african,5d4bf3dfcf0b3a0f3f35ab06,drinking_water,Burkina Faso,alimata,95.05494965
3,west_african,5d4bf3c0cf0b3a0f3f35a892,drinking_water,Burkina Faso,zongo,55.93351031
3,west_african,5d4bef3ecf0b3a0f3f352e38,drinking_water,Nigeria,shemede,124.2390709
3,western,5d4be3edcf0b3a0f3f33f309,kitchen_sink,United States,melissa-s,1796.0
3,western,5d4bde28cf0b3a0f3f335ab3,kitchen_sink,Italy,nazzani,4018.0
3,western,5d4bebcccf0b3a0f3f34ce49,kitchen_sink,United States,hadley,604.0
3,western,5ec4f8c6f0611d7ddd7411af,kitchen_sink,Austria,wessling,1681.0
3,western,5ed6dd80b323617e009eb3ef,kitchen_sink,France,carrillo,7110.0
3,western,5d4be026cf0b3a0f3f338b7c,kitchen_sink,Netherlands,van-beek,3344.0
3,western,5d4be412cf0b3a0f3f33f72e,kitchen_sink,United Kingdom,oliver,1890.0
3,western,5d4be032cf0b3a0f3f338ccc,kitchen_sink,Netherlands,van-kilsdonk,1260.0
3,western,5d4be544cf0b3a0f3f3419e0,kitchen_sink,United States,macintyre,855.0
3,western,6057c01a5ca7db690ba5dcb9,drinking_water,Denmark,hansen-refsing,5343.0
3,western,5ed42132685870b69019a227,drinking_water,France,carrillo,7110.0
3,western,5d4bde2fcf0b3a0f3f335b57,drinking_water,Italy,bravo,3536.0
4,west_african,5d4bf08fcf0b3a0f3f3551e8,water_outlet,Cote d'Ivoire,konan,60.75268981
4,west_african,5d4bf11bcf0b3a0f3f355fe2,water_outlet,Cote d'Ivoire,fofana,41.76707666
4,west_african,5d4bf370cf0b3a0f3f35a05e,get_water,Burkina Faso,mwe-quidraogo,53.64520422
4,west_african,5d4bf0b1cf0b3a0f3f355572,water_outlet,Cote d'Ivoire,yao-bernabi,82.91518231
4,west_african,5d4bf0d6cf0b3a0f3f355930,water_outlet,Cote d'Ivoire,ahova,80.72770904
4,west_african,5d4bf3bdcf0b3a0f3f35a858,water_outlet,Burkina Faso,zongo,55.93351031
4,west_african,5d4bf0f5cf0b3a0f3f355cc8,water_outlet,Cote d'Ivoire,coulibaly,68.11250877
4,west_african,5d4bf3a7cf0b3a0f3f35a666,get_water,Burkina Faso,zongo,55.93351031
4,west_african,5d4bf35ccf0b3a0f3f359e44,get_water,Burkina Faso,sinare,44.71243293
4,west_african,5d4bf0e0cf0b3a0f3f355a56,get_water,Cote d'Ivoire,coulibaly,68.11250877
4,west_african,5d4bf3c6cf0b3a0f3f35a930,get_water,Burkina Faso,alimata,95.05494965
4,west_african,5d4bf105cf0b3a0f3f355d64,get_water,Cote d'Ivoire,fofana,41.76707666
4,west_african,5d4bf37fcf0b3a0f3f35a222,water_outlet,Burkina Faso,mwe-quidraogo,53.64520422
4,west_african,5d4bf45bcf0b3a0f3f35b826,get_water,Burkina Faso,kabore,28.86121301
4,west_african,5d4bdf2acf0b3a0f3f337399,water_outlet,Togo,messan,670.0
4,west_african,5d4bf3ddcf0b3a0f3f35aac0,water_outlet,Burkina Faso,alimata,95.05494965
4,west_african,5d4bf46dcf0b3a0f3f35ba06,water_outlet,Burkina Faso,kabore,28.86121301
4,west_african,5d4bdf32cf0b3a0f3f33745d,water_sources,Togo,ayanou-ahobli,321.0
4,west_african,5d4bf3a0cf0b3a0f3f35a58c,water_outlet,Burkina Faso,bande,52.95427615
4,west_african,5d4bf36bcf0b3a0f3f359fcc,water_outlet,Burkina Faso,sinare,44.71243293
4,west_african,5d4bf07ecf0b3a0f3f35502c,get_water,Cote d'Ivoire,konan,60.75268981
4,west_african,5d4bef0fcf0b3a0f3f3528c2,get_water,Nigeria,ibojionu,72.0
4,west_african,5d4beef7cf0b3a0f3f3525ea,get_water,Nigeria,nwadike,76.0
4,west_african,5d4bef30cf0b3a0f3f352c92,get_water,Nigeria,shemede,124.2390709
4,western,5d4be0d7cf0b3a0f3f339f91,kitchen_sink,Netherlands,jocher,1998.0
4,western,5ec4f8bbf0611d7ddd741160,kitchen_sink,Sweden,elzvik-nystrom,3057.0
4,western,5d4bdfbdcf0b3a0f3f3381bd,kitchen_sink,Netherlands,bogaardt,1540.0
4,western,5ec4f8c6f0611d7ddd7411af,kitchen_sink,Austria,wessling,1681.0
4,western,5d4bde28cf0b3a0f3f335ab3,kitchen_sink,Italy,nazzani,4018.0
4,western,5d4be026cf0b3a0f3f338b7c,kitchen_sink,Netherlands,van-beek,3344.0
4,western,5d4be412cf0b3a0f3f33f72e,kitchen_sink,United Kingdom,oliver,1890.0
4,western,5ec4f54f3f62767d97a47310,water_outlet,France,larriere,19671.0
4,western,5d4be544cf0b3a0f3f3419e0,kitchen_sink,United States,macintyre,855.0
4,western,6057be445ca7db9ba4a5dc09,water_outlet,Denmark,hansen-refsing,5343.0
4,western,5ed421326858707d3119a225,water_outlet,France,carrillo,7110.0
4,western,5d4bde25cf0b3a0f3f335a79,water_outlet,Italy,nazzani,4018.0
5,west_african,5d4bdf2ecf0b3a0f3f337411,grains,Togo,kouleta,45.0
5,west_african,5d4bf0dacf0b3a0f3f3559ae,cooking_utensils,Cote d'Ivoire,ahova,80.72770904
5,west_african,5d4bf3e0cf0b3a0f3f35ab24,grains,Burkina Faso,alimata,95.05494965
5,west_african,5d4bf36ecf0b3a0f3f35a02a,cooking_utensils,Burkina Faso,sinare,44.71243293
5,west_african,5d4bf36dcf0b3a0f3f35a008,grains,Burkina Faso,sinare,44.71243293
5,west_african,5d4bf096cf0b3a0f3f35526a,cooking_utensils,Cote d'Ivoire,konan,60.75268981
5,west_african,5d4bf3c1cf0b3a0f3f35a8b4,grains,Burkina Faso,zongo,55.93351031
5,west_african,5d4bf0b8cf0b3a0f3f355636,cooking_utensils,Cote d'Ivoire,yao-bernabi,82.91518231
5,west_african,5d4bdf24cf0b3a0f3f3372f1,grains,Togo,tchamou,145.0
5,west_african,5d4bf122cf0b3a0f3f356042,grains,Cote d'Ivoire,fofana,41.76707666
5,west_african,5d4bf3c2cf0b3a0f3f35a8c8,cooking_utensils,Burkina Faso,zongo,55.93351031
5,west_african,5d4bf46fcf0b3a0f3f35ba48,grains,Burkina Faso,kabore,28.86121301
5,west_african,5d4bdf33cf0b3a0f3f337473,cooking_utensils,Togo,ayanou-ahobli,321.0
5,west_african,5d4bf3e1cf0b3a0f3f35ab3a,cooking_utensils,Burkina Faso,alimata,95.05494965
5,west_african,5d4bf471cf0b3a0f3f35ba68,cooking_utensils,Burkina Faso,kabore,28.86121301
5,west_african,5d4bf0fecf0b3a0f3f355d26,cooking_utensils,Cote d'Ivoire,coulibaly,68.11250877
5,west_african,5d4bdf29cf0b3a0f3f337385,grains,Togo,messan,670.0
5,west_african,5d4bf382cf0b3a0f3f35a272,cooking_utensils,Burkina Faso,mwe-quidraogo,53.64520422
5,west_african,5d4bf3a5cf0b3a0f3f35a61e,cooking_utensils,Burkina Faso,bande,52.95427615
5,west_african,5d4bf0dacf0b3a0f3f355994,grains,Cote d'Ivoire,ahova,80.72770904
5,west_african,5d4be043cf0b3a0f3f338e9a,grains,Nigeria,ojah,1643.0
5,west_african,5d4beefccf0b3a0f3f35267a,grains,Nigeria,nwadike,76.0
5,west_african,5d4bef02cf0b3a0f3f352736,cooking_utensils,Nigeria,nwadike,76.0
5,west_african,5d4bef10cf0b3a0f3f3528c8,grains,Nigeria,ibojionu,72.0
5,western,5d4bde2fcf0b3a0f3f335b61,grains,Italy,bravo,3536.0
5,western,5ec4f54d3f62767d97a47300,grains,France,larriere,19671.0
5,western,5d4bde91cf0b3a0f3f336469,grains,Netherlands,rasing,3150.0
5,western,5ed419e36858709b5b19a209,cooking_utensils,France,carrillo,7110.0
5,western,5d4bdf9fcf0b3a0f3f337f1f,cooking_utensils,Switzerland,renato,2602.0
5,western,5ec4f93ef0611d7ddd7415ae,cooking_utensils,Spain,tejada,2838.0
5,western,5ec4f8c9f0611d7ddd7411c6,cooking_utensils,Sweden,elzvik-nystrom,3057.0
5,western,5d4be3f4cf0b3a0f3f33f3d5,cooking_utensils,United States,melissa-s,1796.0
5,western,5ec4f905f0611d7ddd7413c1,grains,Spain,dela-torre-sawtana,4553.0
5,western,5d4be1efcf0b3a0f3f33bdaf,cooking_utensils,United States,hoffman-tsuhlares,2347.0
5,western,5d4be093cf0b3a0f3f3397bd,grains,Netherlands,chang,1728.0
5,western,5d4be53acf0b3a0f3f3418aa,cooking_utensils,United States,diamond-01,2001.0
6,west_african,5d4bdf38cf0b3a0f3f337511,plate_of_food,Togo,ayanou-ahobli,321.0
6,west_african,5d4bf0ddcf0b3a0f3f3559f4,place_where_eating_dinner,Cote d'Ivoire,ahova,80.72770904
6,west_african,5d4bdf21cf0b3a0f3f3372ad,table_with_food,Togo,tchamou,145.0
6,west_african,5d4bf36fcf0b3a0f3f35a056,place_where_eating_dinner,Burkina Faso,sinare,44.71243293
6,west_african,5d4bdf22cf0b3a0f3f3372c3,plate_of_food,Togo,tchamou,145.0
6,west_african,5d4bf099cf0b3a0f3f3552ae,place_where_eating_dinner,Cote d'Ivoire,konan,60.75268981
6,west_african,5d4bdf27cf0b3a0f3f33734d,plate_of_food,Togo,messan,670.0
6,west_african,5d4bf0b4cf0b3a0f3f3555d6,place_where_eating_dinner,Cote d'Ivoire,yao-bernabi,82.91518231
6,west_african,5d4bdf26cf0b3a0f3f33732d,table_with_food,Togo,messan,670.0
6,west_african,5d4bf3a6cf0b3a0f3f35a63a,plate_of_food,Burkina Faso,bande,52.95427615
6,west_african,5d4bf3c5cf0b3a0f3f35a91e,place_where_eating_dinner,Burkina Faso,zongo,55.93351031
6,west_african,5d4bdf2ccf0b3a0f3f3373e3,plate_of_food,Togo,kouleta,45.0
6,west_african,5d4bdf26cf0b3a0f3f33732d,place_where_eating_dinner,Togo,messan,670.0
6,west_african,5d4bf3e3cf0b3a0f3f35ab6c,place_where_eating_dinner,Burkina Faso,alimata,95.05494965
6,west_african,5d4bf124cf0b3a0f3f356082,place_where_eating_dinner,Cote d'Ivoire,fofana,41.76707666
6,west_african,5d4bf104cf0b3a0f3f355d52,place_where_eating_dinner,Cote d'Ivoire,coulibaly,68.11250877
6,west_african,5d4bf472cf0b3a0f3f35ba8e,place_where_eating_dinner,Burkina Faso,kabore,28.86121301
6,west_african,5d4bf385cf0b3a0f3f35a2c0,place_where_eating_dinner,Burkina Faso,mwe-quidraogo,53.64520422
6,west_african,5d4bf3a7cf0b3a0f3f35a65a,place_where_eating_dinner,Burkina Faso,bande,52.95427615
6,west_african,5d4bdf2bcf0b3a0f3f3373d1,table_with_food,Togo,kouleta,45.0
6,west_african,5d4bf056cf0b3a0f3f354c10,table_with_food,Liberia,doe,87.0
6,west_african,5d4bef16cf0b3a0f3f352984,place_where_eating_dinner,Nigeria,ibojionu,72.0
6,west_african,5d4bef0ecf0b3a0f3f3528a6,plate_of_food,Nigeria,ibojionu,72.0
6,west_african,5d4be012cf0b3a0f3f338a68,table_with_food,Nigeria,amama,3750.0
6,western,5d4be4ebcf0b3a0f3f340f94,plate_of_food,United States,williams,2055.0
6,western,5d4be1dbcf0b3a0f3f33bb93,table_with_food,United States,hoffman-tsuhlares,2347.0
6,western,5f1700444a4b3f1400a21c14,plate_of_food,United States,moore,1732.0
6,western,5ec4f84bf0611d7ddd740d7d,table_with_food,United States,deveney,5125.0
6,western,5d4bed30cf0b3a0f3f34f254,table_with_food,United States,grabowski,2041.0
6,western,5d4bde8dcf0b3a0f3f336417,plate_of_food,Netherlands,rasing,3150.0
6,western,5ed6c62ab3236122f49eb39a,plate_of_food,France,carrillo,7110.0
6,western,5d4bde21cf0b3a0f3f335a21,plate_of_food,Italy,nazzani,4018.0
6,western,5ec4f8f0f0611d7ddd74131c,plate_of_food,Spain,garrido,4253.0
6,western,5d4bde8bcf0b3a0f3f3363e3,table_with_food,Netherlands,rasing,3150.0
6,western,5d4be545cf0b3a0f3f341a04,table_with_food,United States,macintyre,855.0
6,western,5ec4fc31f0611d7ddd742e71,table_with_food,United States,stringini,960.0
7,west_african,5d4bf37bcf0b3a0f3f35a1b4,cleaning_equipment,Burkina Faso,mwe-quidraogo,53.64520422
7,west_african,5d4bf0d2cf0b3a0f3f3558b4,cleaning_equipment,Cote d'Ivoire,ahova,80.72770904
7,west_african,5d4bf0f0cf0b3a0f3f355c38,cleaning_equipment,Cote d'Ivoire,coulibaly,68.11250877
7,west_african,5d4bdf2fcf0b3a0f3f337427,cleaning_equipment,Togo,kouleta,45.0
7,west_african,5d4bdf2acf0b3a0f3f3373ab,cleaning_equipment,Togo,messan,670.0
7,west_african,5d4bf3b6cf0b3a0f3f35a7d4,cleaning_equipment,Burkina Faso,zongo,55.93351031
7,west_african,5d4bf116cf0b3a0f3f355f48,cleaning_equipment,Cote d'Ivoire,fofana,41.76707666
7,west_african,5d4bdf25cf0b3a0f3f33730f,cleaning_floors,Togo,tchamou,145.0
7,west_african,5d4bf469cf0b3a0f3f35b990,cleaning_equipment,Burkina Faso,kabore,28.86121301
7,west_african,5d4bdf2acf0b3a0f3f3373ab,cleaning_floors,Togo,messan,670.0
7,west_african,5d4bf3d7cf0b3a0f3f35aa1a,cleaning_equipment,Burkina Faso,alimata,95.05494965
7,west_african,5d4bf0accf0b3a0f3f3554d2,cleaning_equipment,Cote d'Ivoire,yao-bernabi,82.91518231
7,west_african,5d4bdf2fcf0b3a0f3f337427,cleaning_floors,Togo,kouleta,45.0
7,west_african,5d4bf08acf0b3a0f3f355162,cleaning_equipment,Cote d'Ivoire,konan,60.75268981
7,west_african,5d4bef14cf0b3a0f3f35294c,cleaning_equipment,Nigeria,ibojionu,72.0
7,west_african,5d4bef3dcf0b3a0f3f352e22,cleaning_equipment,Nigeria,shemede,124.2390709
7,west_african,5d4bf05ccf0b3a0f3f354ca8,cleaning_equipment,Liberia,blamoh,116.0
7,west_african,5d4beeeacf0b3a0f3f352462,cleaning_equipment,Nigeria,oputa-nwadioro,67.67970667
7,west_african,5d4be00ecf0b3a0f3f338a14,cleaning_equipment,Nigeria,amama,3750.0
7,west_african,5d4beefdcf0b3a0f3f3526a2,cleaning_equipment,Nigeria,nwadike,76.0
7,west_african,5d4bef27cf0b3a0f3f352b94,cleaning_equipment,Nigeria,ehegwo,59.0
7,west_african,5d4be004cf0b3a0f3f338938,cleaning_floors,Nigeria,laoye,725.0
7,west_african,5d4bf04ccf0b3a0f3f354b42,cleaning_equipment,Liberia,nyanford,48.0
7,west_african,5d4be00ecf0b3a0f3f338a14,cleaning_floors,Nigeria,amama,3750.0
7,western,5d4bde5acf0b3a0f3f335f4d,cleaning_floors,United States,stringini,960.0
7,western,5ec4fbf0f0611d7ddd742c57,cleaning_floors,France,rayez,875.0
7,western,5d4bdf9fcf0b3a0f3f337f27,cleaning_floors,Switzerland,renato,2602.0
7,western,5d4be408cf0b3a0f3f33f60c,cleaning_floors,United Kingdom,barnard,2010.0
7,western,5f184bb76d14e3e1c24de0b2,cleaning_equipment,United States,moore,1732.0
7,western,5d4be03acf0b3a0f3f338da2,cleaning_equipment,Netherlands,van-der-burgt,1816.0
7,western,5d4be094cf0b3a0f3f3397d7,cleaning_floors,Netherlands,chang,1728.0
7,western,5d4be417cf0b3a0f3f33f7c6,cleaning_floors,United Kingdom,oliver,1890.0
7,western,5ec4f907f0611d7ddd7413db,cleaning_floors,Spain,dela-torre-sawtana,4553.0
7,western,5d4bdfb7cf0b3a0f3f33812d,cleaning_floors,Netherlands,peters,1038.0
7,western,5ec4f930f0611d7ddd74153e,cleaning_equipment,Spain,juan-caballero,7639.0
7,western,5d4be4eccf0b3a0f3f340fb0,cleaning_floors,United States,williams,2055.0
8,west_african,5d4bf3cccf0b3a0f3f35a974,sitting_area,Burkina Faso,alimata,95.05494965
8,west_african,5d4bf0e4cf0b3a0f3f355abe,sitting_area,Cote d'Ivoire,coulibaly,68.11250877
8,west_african,5d4bf084cf0b3a0f3f3550d4,armchair,Cote d'Ivoire,konan,60.75268981
8,west_african,5d4bf371cf0b3a0f3f35a088,sitting_area,Burkina Faso,mwe-quidraogo,53.64520422
8,west_african,5d4bf0bdcf0b3a0f3f3556c2,sitting_area,Cote d'Ivoire,yao-bernabi,82.91518231
8,west_african,5d4bf099cf0b3a0f3f3552b8,sitting_area,Cote d'Ivoire,konan,60.75268981
8,west_african,5d4bf35dcf0b3a0f3f359e68,sitting_area,Burkina Faso,sinare,44.71243293
8,west_african,5d4bf099cf0b3a0f3f3552b8,armchair,Cote d'Ivoire,konan,60.75268981
8,west_african,5d4bdf2dcf0b3a0f3f3373ed,armchair,Togo,kouleta,45.0
8,west_african,5d4bf0c1cf0b3a0f3f355730,sitting_area,Cote d'Ivoire,ahova,80.72770904
8,west_african,5d4bdf35cf0b3a0f3f3374ab,armchair,Togo,ayanou-ahobli,321.0
8,west_african,5d4bdf27cf0b3a0f3f33735f,armchair,Togo,messan,670.0
8,west_african,5d4bf35dcf0b3a0f3f359e68,armchair,Burkina Faso,sinare,44.71243293
8,west_african,5d4bf3a8cf0b3a0f3f35a678,sitting_area,Burkina Faso,zongo,55.93351031
8,west_african,5d4bf114cf0b3a0f3f355efc,armchair,Cote d'Ivoire,fofana,41.76707666
8,west_african,5d4bdf27cf0b3a0f3f33735f,sitting_area,Togo,messan,670.0
8,west_african,5d4bdf24cf0b3a0f3f3372f3,sitting_area,Togo,tchamou,145.0
8,west_african,5d4bf107cf0b3a0f3f355da0,sitting_area,Cote d'Ivoire,fofana,41.76707666
8,west_african,5d4bf45fcf0b3a0f3f35b866,sitting_area,Burkina Faso,kabore,28.86121301
8,west_african,5d4bf38fcf0b3a0f3f35a3ec,sitting_area,Burkina Faso,bande,52.95427615
8,west_african,5d4bdf2dcf0b3a0f3f3373ed,sitting_area,Togo,kouleta,45.0
8,west_african,5d4bdf24cf0b3a0f3f3372f3,armchair,Togo,tchamou,145.0
8,west_african,5ec4facbf0611d7ddd7422b4,sitting_area,Ghana,nkrumah,375.0
8,west_african,5ec4fb0ef0611d7ddd7424ea,sitting_area,Ghana,ampiah,213.0
8,western,5d4bde23cf0b3a0f3f335a49,couch,Italy,nazzani,4018.0
8,western,605a669c801a8781d885e8da,couch,Denmark,hansen-refsing,5343.0
8,western,5ec4fc43f0611d7ddd742f0a,couch,France,chamaitte,803.0
8,western,5d4bed30cf0b3a0f3f34f260,couch,United States,grabowski,2041.0
8,western,5d4be53fcf0b3a0f3f341936,couch,United States,macintyre,855.0
8,western,5ec4f8a2f0611d7ddd74107b,couch,Sweden,elzvik-nystrom,3057.0
8,western,5d4be1e7cf0b3a0f3f33bcc9,couch,United States,hoffman-tsuhlares,2347.0
8,western,5d4be026cf0b3a0f3f338b88,couch,Netherlands,van-beek,3344.0
8,western,5d4be0d9cf0b3a0f3f339fbd,couch,Netherlands,ploum,2370.0
8,western,5d4be4e7cf0b3a0f3f340f20,couch,United States,williams,2055.0
8,western,5ec4f8eef0611d7ddd741308,couch,Spain,garrido,4253.0
8,western,5ec4f8fcf0611d7ddd741384,couch,Spain,dela-torre-sawtana,4553.0
9,west_african,5d4bf0f0cf0b3a0f3f355c26,bed,Cote d'Ivoire,coulibaly,68.11250877
9,west_african,5d4bf089cf0b3a0f3f355148,bed,Cote d'Ivoire,konan,60.75268981
9,west_african,5d4bf0cdcf0b3a0f3f35587e,bed,Cote d'Ivoire,ahova,80.72770904
9,west_african,5d4bf115cf0b3a0f3f355f2e,bed,Cote d'Ivoire,fofana,41.76707666
9,west_african,5d4bf3d6cf0b3a0f3f35aa00,bed,Burkina Faso,alimata,95.05494965
9,west_african,5d4bf37acf0b3a0f3f35a198,bed,Burkina Faso,mwe-quidraogo,53.64520422
9,west_african,5d4bf0d1cf0b3a0f3f3558a6,bed,Cote d'Ivoire,ahova,80.72770904
9,west_african,5d4bf0aacf0b3a0f3f355498,bed,Cote d'Ivoire,yao-bernabi,82.91518231
9,west_african,5d4bf366cf0b3a0f3f359f40,bed,Burkina Faso,sinare,44.71243293
9,west_african,5d4bf115cf0b3a0f3f355f20,bed,Cote d'Ivoire,fofana,41.76707666
9,west_african,5d4bf0aacf0b3a0f3f3554a0,bed,Cote d'Ivoire,yao-bernabi,82.91518231
9,west_african,5d4bdf30cf0b3a0f3f33742f,bed,Togo,kouleta,45.0
9,west_african,5d4bf0d1cf0b3a0f3f35589a,bed,Cote d'Ivoire,ahova,80.72770904
9,west_african,5d4bf088cf0b3a0f3f35512e,bed,Cote d'Ivoire,konan,60.75268981
9,west_african,5d4bf39acf0b3a0f3f35a4dc,bed,Burkina Faso,bande,52.95427615
9,west_african,5d4bf3d6cf0b3a0f3f35aa04,bed,Burkina Faso,alimata,95.05494965
9,west_african,5d4bf468cf0b3a0f3f35b97c,bed,Burkina Faso,kabore,28.86121301
9,west_african,5d4bf0efcf0b3a0f3f355c1a,bed,Cote d'Ivoire,coulibaly,68.11250877
9,west_african,5d4bf468cf0b3a0f3f35b974,bed,Burkina Faso,kabore,28.86121301
9,west_african,5d4bf0efcf0b3a0f3f355c10,bed,Cote d'Ivoire,coulibaly,68.11250877
9,west_african,5d4bdf2bcf0b3a0f3f3373b5,bed,Togo,messan,670.0
9,west_african,5d4bdf34cf0b3a0f3f337491,bed,Togo,ayanou-ahobli,321.0
9,west_african,5d4bf37acf0b3a0f3f35a190,bed,Burkina Faso,mwe-quidraogo,53.64520422
9,west_african,5d4bf39acf0b3a0f3f35a4e2,bed,Burkina Faso,bande,52.95427615
9,western,5ec4f908f0611d7ddd7413e2,bed,Spain,dela-torre-sawtana,4553.0
9,western,5d4be407cf0b3a0f3f33f5ee,bed,United Kingdom,barnard,2010.0
9,western,5d4bdf9fcf0b3a0f3f337f31,bed,Switzerland,renato,2602.0
9,western,5d4bde22cf0b3a0f3f335a37,bed,Italy,nazzani,4018.0
9,western,5d4be025cf0b3a0f3f338b6c,bed,Netherlands,van-beek,3344.0
9,western,6057d4085ca47c5cc1e756ca,bed,Denmark,hansen-refsing,5343.0
9,western,5ec4f858f0611d7ddd740df1,bed,United States,deveney,5125.0
9,western,5ec4f940f0611d7ddd7415b8,bed,Spain,tejada,2838.0
9,western,5d4be418cf0b3a0f3f33f7ec,bed,United Kingdom,oliver,1890.0
9,western,5d4be0d2cf0b3a0f3f339f01,bed,Netherlands,jocher,1998.0
9,western,5d4bde94cf0b3a0f3f3364b7,bed,Netherlands,rasing,3150.0
9,western,5ec4f908f0611d7ddd7413e3,bed,Spain,dela-torre-sawtana,4553.0
10,west_african,5d4bf118cf0b3a0f3f355f82,toilet,Cote d'Ivoire,fofana,41.76707666
10,west_african,5d4bf46ccf0b3a0f3f35b9f0,toilet,Burkina Faso,kabore,28.86121301
10,west_african,5d4bf0d4cf0b3a0f3f3558f2,toilet,Cote d'Ivoire,ahova,80.72770904
10,west_african,5d4bf3b8cf0b3a0f3f35a814,toilet,Burkina Faso,zongo,55.93351031
10,west_african,5d4bf07ecf0b3a0f3f35502a,toilet,Cote d'Ivoire,konan,60.75268981
10,west_african,5d4bdf36cf0b3a0f3f3374cf,toilet,Togo,ayanou-ahobli,321.0
10,west_african,5d4bf3b7cf0b3a0f3f35a7fc,toilet,Burkina Faso,zongo,55.93351031
10,west_african,5d4bf3d9cf0b3a0f3f35aa54,toilet,Burkina Faso,alimata,95.05494965
10,west_african,5d4bf3d9cf0b3a0f3f35aa4a,toilet,Burkina Faso,alimata,95.05494965
10,west_african,5d4bf0aecf0b3a0f3f35550e,toilet,Cote d'Ivoire,yao-bernabi,82.91518231
10,west_african,5d4bdf2bcf0b3a0f3f3373b7,toilet,Togo,messan,670.0
10,west_african,5d4bf0adcf0b3a0f3f3554fe,toilet,Cote d'Ivoire,yao-bernabi,82.91518231
10,west_african,5d4bf08ccf0b3a0f3f355196,toilet,Cote d'Ivoire,konan,60.75268981
10,west_african,5d4bf37dcf0b3a0f3f35a1da,toilet,Burkina Faso,mwe-quidraogo,53.64520422
10,west_african,5d4bf0adcf0b3a0f3f3554f4,toilet,Cote d'Ivoire,yao-bernabi,82.91518231
10,west_african,5d4bf0d4cf0b3a0f3f3558e6,toilet,Cote d'Ivoire,ahova,80.72770904
10,west_african,5d4bf3dacf0b3a0f3f35aa68,toilet,Burkina Faso,alimata,95.05494965
10,west_african,5d4bdf32cf0b3a0f3f337465,toilet,Togo,ayanou-ahobli,321.0
10,west_african,5d4bf118cf0b3a0f3f355f88,toilet,Cote d'Ivoire,fofana,41.76707666
10,west_african,5d4bf08ccf0b3a0f3f3551a2,toilet,Cote d'Ivoire,konan,60.75268981
10,west_african,5d4bf39dcf0b3a0f3f35a536,toilet,Burkina Faso,bande,52.95427615
10,west_african,5d4bdf2bcf0b3a0f3f3373c5,toilet,Togo,kouleta,45.0
10,west_african,5d4bdf25cf0b3a0f3f337315,toilet,Togo,tchamou,145.0
10,west_african,5d4bf0f3cf0b3a0f3f355c7e,toilet,Cote d'Ivoire,coulibaly,68.11250877
10,western,5efb663a48cdebbec9b6300c,toilet,Austria,posch,3003.0
10,western,5d4be04bcf0b3a0f3f338f7c,toilet,Netherlands,chang,1728.0
10,western,5d4be53ccf0b3a0f3f3418e4,toilet,United States,diamond-01,2001.0
10,western,5d4bdebfcf0b3a0f3f336827,toilet,United States,duncan,800.0
10,western,5f17731e9607c1941974c2ca,toilet,United States,moore,1732.0
10,western,5ec4f869f0611d7ddd740e92,toilet,United States,deveney,5125.0
10,western,5d4be403cf0b3a0f3f33f572,toilet,United Kingdom,barnard,2010.0
10,western,5d4bde55cf0b3a0f3f335ed7,toilet,United States,stringini,960.0
10,western,6057a174684cb26989ce4087,toilet,Denmark,hansen-refsing,5343.0
10,western,5ec4f84af0611d7ddd740d79,toilet,United States,deveney,5125.0
10,western,5d4be051cf0b3a0f3f33901e,toilet,Netherlands,ploum,2370.0
10,western,5d4be095cf0b3a0f3f3397ff,toilet,United States,janacek,4559.0
11,west_african,5d4bf384cf0b3a0f3f35a2ba,light_source_in_kitchen,Burkina Faso,mwe-quidraogo,53.64520422
11,west_african,5d4bf0cccf0b3a0f3f35586e,light_source_in_livingroom,Cote d'Ivoire,ahova,80.72770904
11,west_african,5d4bf0dccf0b3a0f3f3559e0,light_source_in_kitchen,Cote d'Ivoire,ahova,80.72770904
11,west_african,5d4bf114cf0b3a0f3f355f02,light_source_in_livingroom,Cote d'Ivoire,fofana,41.76707666
11,west_african,5d4bf3a7cf0b3a0f3f35a652,light_source_in_kitchen,Burkina Faso,bande,52.95427615
11,west_african,5d4bf07fcf0b3a0f3f35504a,light_source_in_livingroom,Cote d'Ivoire,konan,60.75268981
11,west_african,5d4bf3c5cf0b3a0f3f35a914,light_source_in_kitchen,Burkina Faso,zongo,55.93351031
11,west_african,5d4bf0a8cf0b3a0f3f355470,light_source_in_livingroom,Cote d'Ivoire,yao-bernabi,82.91518231
11,west_african,5d4bf0b4cf0b3a0f3f3555cc,light_source_in_kitchen,Cote d'Ivoire,yao-bernabi,82.91518231
11,west_african,5d4bf36fcf0b3a0f3f35a04e,light_source_in_kitchen,Burkina Faso,sinare,44.71243293
11,west_african,5d4bf379cf0b3a0f3f35a172,light_source_in_livingroom,Burkina Faso,mwe-quidraogo,53.64520422
11,west_african,5d4bf124cf0b3a0f3f35607c,light_source_in_kitchen,Cote d'Ivoire,fofana,41.76707666
11,west_african,5d4bdf2acf0b3a0f3f3373ad,light_source_in_livingroom,Togo,messan,670.0
11,west_african,5d4bf3d5cf0b3a0f3f35a9ec,light_source_in_livingroom,Burkina Faso,alimata,95.05494965
11,west_african,5d4bdf28cf0b3a0f3f337365,light_source_in_kitchen,Togo,messan,670.0
11,west_african,5d4bf0eecf0b3a0f3f355bfa,light_source_in_livingroom,Cote d'Ivoire,coulibaly,68.11250877
11,west_african,5d4bf3e3cf0b3a0f3f35ab66,light_source_in_kitchen,Burkina Faso,alimata,95.05494965
11,west_african,5d4bf399cf0b3a0f3f35a4c8,light_source_in_livingroom,Burkina Faso,bande,52.95427615
11,west_african,5d4bf365cf0b3a0f3f359f30,light_source_in_livingroom,Burkina Faso,sinare,44.71243293
11,west_african,5d4bf098cf0b3a0f3f3552a4,light_source_in_kitchen,Cote d'Ivoire,konan,60.75268981
11,west_african,5d4be007cf0b3a0f3f33896a,light_source_in_kitchen,Nigeria,laoye,725.0
11,west_african,5d4beef2cf0b3a0f3f352564,light_source_in_livingroom,Nigeria,oputa-nwadioro,67.67970667
11,west_african,5d4bef23cf0b3a0f3f352b14,light_source_in_livingroom,Nigeria,ehegwo,59.0
11,west_african,5d4be089cf0b3a0f3f339683,light_source_in_kitchen,Nigeria,oke,277.0
11,western,5d4be4e8cf0b3a0f3f340f24,light_source_in_livingroom,United States,williams,2055.0
11,western,5d4be3eccf0b3a0f3f33f2ef,light_source_in_livingroom,United States,melissa-s,1796.0
11,western,5ec4f54c3f62767d97a472f4,light_source_in_kitchen,France,larriere,19671.0
11,western,5d4be412cf0b3a0f3f33f724,light_sources,United Kingdom,oliver,1890.0
11,western,5d4bde59cf0b3a0f3f335f2f,light_source_in_kitchen,United States,stringini,960.0
11,western,5ec4f903f0611d7ddd7413b7,light_source_in_livingroom,Spain,dela-torre-sawtana,4553.0
11,western,5ec4f837f0611d7ddd740cdd,light_source_in_kitchen,United States,rivera,3478.0
11,western,5d4be1e8cf0b3a0f3f33bcd3,light_source_in_kitchen,United States,hoffman-tsuhlares,2347.0
11,western,5d4be034cf0b3a0f3f338cfa,light_source_in_livingroom,Netherlands,van-der-burgt,1816.0
11,western,5ec4f910f0611d7ddd741429,light_source_in_livingroom,Spain,sanchez,3195.0
11,western,5d4bde2acf0b3a0f3f335adf,light_source_in_livingroom,Italy,bravo,3536.0
11,western,5d4be039cf0b3a0f3f338d88,light_source_in_kitchen,Netherlands,van-der-burgt,1816.0
12,west_african,5d4bdf2ecf0b3a0f3f337403,home,Togo,kouleta,45.0
12,west_african,5d4bdf23cf0b3a0f3f3372e5,home,Togo,tchamou,145.0
12,west_african,5d4bf3abcf0b3a0f3f35a6c0,home,Burkina Faso,zongo,55.93351031
12,west_african,5d4bf360cf0b3a0f3f359e9a,home,Burkina Faso,sinare,44.71243293
12,west_african,5d4bf373cf0b3a0f3f35a0be,home,Burkina Faso,mwe-quidraogo,53.64520422
12,west_african,5d4bf0c3cf0b3a0f3f355776,home,Cote d'Ivoire,ahova,80.72770904
12,west_african,5d4bdf2fcf0b3a0f3f33741d,home,Togo,kouleta,45.0
12,west_african,5d4bf10acf0b3a0f3f355df0,home,Cote d'Ivoire,fofana,41.76707666
12,west_african,5d4bdf24cf0b3a0f3f3372fd,home,Togo,tchamou,145.0
12,west_african,5d4bf461cf0b3a0f3f35b8ac,home,Burkina Faso,kabore,28.86121301
12,west_african,5d4bdf27cf0b3a0f3f33734b,home,Togo,messan,670.0
12,west_african,5d4bdf34cf0b3a0f3f33749b,home,Togo,ayanou-ahobli,321.0
12,west_african,5d4bf0dfcf0b3a0f3f355a3e,home,Cote d'Ivoire,coulibaly,68.11250877
12,west_african,5d4bf080cf0b3a0f3f355060,home,Cote d'Ivoire,konan,60.75268981
12,west_african,5d4bf3cdcf0b3a0f3f35a97c,home,Burkina Faso,alimata,95.05494965
12,west_african,5d4bf0bacf0b3a0f3f355672,home,Cote d'Ivoire,yao-bernabi,82.91518231
12,west_african,5d4bf391cf0b3a0f3f35a418,home,Burkina Faso,bande,52.95427615
12,west_african,5d4beefacf0b3a0f3f352658,home,Nigeria,nwadike,76.0
12,west_african,5d4bef28cf0b3a0f3f352ba4,home,Nigeria,ehegwo,59.0
12,west_african,5d4be007cf0b3a0f3f338970,home,Nigeria,laoye,725.0
12,west_african,5d4bf050cf0b3a0f3f354ba8,home,Liberia,galrbah,58.0
12,west_african,5d4be00acf0b3a0f3f3389be,home,Nigeria,laoye,725.0
12,west_african,5d4bf055cf0b3a0f3f354bf4,home,Liberia,doe,87.0
12,west_african,5d4be03ecf0b3a0f3f338e02,home,Nigeria,ojah,1643.0
12,western,5d4bdfb5cf0b3a0f3f338107,home,Netherlands,peters,1038.0
12,western,5ec4f91ff0611d7ddd7414a7,home,Spain,rodrigo,4256.0
12,western,5d4bde2bcf0b3a0f3f335afd,home,Italy,bravo,3536.0
12,western,5d4be407cf0b3a0f3f33f5dc,home,United Kingdom,barnard,2010.0
12,western,5ec4f54d3f62767d97a472fa,home,France,larriere,19671.0
12,western,6057d3045ca47c7370e7565d,home,Denmark,hansen-refsing,5343.0
12,western,5d4bed33cf0b3a0f3f34f2ac,home,United States,grabowski,2041.0
12,western,5d4be050cf0b3a0f3f33900c,home,Netherlands,ploum,2370.0
12,western,5d4bde91cf0b3a0f3f33646b,home,Netherlands,rasing,3150.0
12,western,5fc93fb29216a10ceba3585e,home,United States,theresa-s,3450.0
12,western,5ed6c62ab323615ce59eb395,home,France,carrillo,7110.0
12,western,5d4be41dcf0b3a0f3f33f888,home,United States,diamond,4446.0
13,west_african,5d4bdf2ccf0b3a0f3f3373df,roof,Togo,kouleta,45.0
13,west_african,5d4bf09ccf0b3a0f3f35530a,roof,Cote d'Ivoire,konan,60.75268981
13,west_african,5d4bf3a8cf0b3a0f3f35a67c,roof,Burkina Faso,zongo,55.93351031
13,west_african,5d4bf35dcf0b3a0f3f359e6c,roof,Burkina Faso,sinare,44.71243293
13,west_african,5d4bf0bdcf0b3a0f3f3556c6,roof,Cote d'Ivoire,yao-bernabi,82.91518231
13,west_african,5d4bf390cf0b3a0f3f35a3f8,roof,Burkina Faso,bande,52.95427615
13,west_african,5d4bf107cf0b3a0f3f355daa,roof,Cote d'Ivoire,fofana,41.76707666
13,west_african,5d4bdf34cf0b3a0f3f33749f,roof,Togo,ayanou-ahobli,321.0
13,west_african,5d4bf45fcf0b3a0f3f35b874,roof,Burkina Faso,kabore,28.86121301
13,west_african,5d4bdf26cf0b3a0f3f337339,roof,Togo,messan,670.0
13,west_african,5d4bf0bfcf0b3a0f3f355712,roof,Cote d'Ivoire,ahova,80.72770904
13,west_african,5d4bf0e4cf0b3a0f3f355ac6,roof,Cote d'Ivoire,coulibaly,68.11250877
13,west_african,5d4bdf22cf0b3a0f3f3372bf,roof,Togo,tchamou,145.0
13,west_african,5d4bf3e4cf0b3a0f3f35ab7a,roof,Burkina Faso,alimata,95.05494965
13,west_african,5d4bf372cf0b3a0f3f35a092,roof,Burkina Faso,mwe-quidraogo,53.64520422
13,west_african,5d4bef1ecf0b3a0f3f352a78,roof,Nigeria,ehegwo,59.0
13,west_african,5d4beee7cf0b3a0f3f35240c,roof,Nigeria,oputa-nwadioro,67.67970667
13,west_african,5d4bef13cf0b3a0f3f35293c,roof,Nigeria,ibojionu,72.0
13,west_african,5ec4fad3f0611d7ddd7422fd,roof,Ghana,nkrumah,375.0
13,west_african,5d4bf04fcf0b3a0f3f354b90,roof,Liberia,galrbah,58.0
13,west_african,5d4bf056cf0b3a0f3f354c04,roof,Liberia,doe,87.0
13,west_african,5d4bef06cf0b3a0f3f3527b2,roof,Nigeria,udemba,41.0
13,west_african,5d4bf05bcf0b3a0f3f354c82,roof,Liberia,blamoh,116.0
13,west_african,5d4be014cf0b3a0f3f338a96,roof,Nigeria,amama,3750.0
13,western,5d4be540cf0b3a0f3f341960,roof,United States,macintyre,855.0
13,western,5ec4fc35f0611d7ddd742e94,roof,France,chamaitte,803.0
13,western,5ec4f934f0611d7ddd741567,roof,Spain,tejada,2838.0
13,western,5ed6d108b32361dc379eb3c6,roof,France,carrillo,7110.0
13,western,5d4be535cf0b3a0f3f341818,roof,United States,diamond-01,2001.0
13,western,5d4be3d0cf0b3a0f3f33ef9b,roof,United States,theresa-s,3450.0
13,western,5d4be0dacf0b3a0f3f339fdf,roof,Netherlands,ploum,2370.0
13,western,5ec4f8fff0611d7ddd74138f,roof,Spain,dela-torre-sawtana,4553.0
13,western,5d4bde2bcf0b3a0f3f335b03,roof,Italy,bravo,3536.0
13,western,5d4bed31cf0b3a0f3f34f278,roof,United States,grabowski,2041.0
13,western,5d4be1dacf0b3a0f3f33bb71,roof,United States,hoffman-tsuhlares,2347.0
13,western,5ec4f919f0611d7ddd741476,roof,Spain,rodrigo,4256.0
14,west_african,5d4bdf34cf0b3a0f3f337497,front_door,Togo,ayanou-ahobli,321.0
14,west_african,5d4bf109cf0b3a0f3f355dd2,front_door,Cote d'Ivoire,fofana,41.76707666
14,west_african,5d4bf0c3cf0b3a0f3f35576e,front_door,Cote d'Ivoire,ahova,80.72770904
14,west_african,5d4bf3e5cf0b3a0f3f35aba2,front_door,Burkina Faso,alimata,95.05494965
14,west_african,5d4bf3e5cf0b3a0f3f35ab98,front_door,Burkina Faso,alimata,95.05494965
14,west_african,5d4bf0becf0b3a0f3f3556e8,front_door,Cote d'Ivoire,yao-bernabi,82.91518231
14,west_african,5d4bf0becf0b3a0f3f3556de,front_door,Cote d'Ivoire,yao-bernabi,82.91518231
14,west_african,5d4bf0c3cf0b3a0f3f355764,front_door,Cote d'Ivoire,ahova,80.72770904
14,west_african,5d4bf372cf0b3a0f3f35a0a0,front_door,Burkina Faso,mwe-quidraogo,53.64520422
14,west_african,5d4bdf24cf0b3a0f3f3372f9,front_door,Togo,tchamou,145.0
14,west_african,5d4bf0e7cf0b3a0f3f355b1e,front_door,Cote d'Ivoire,coulibaly,68.11250877
14,west_african,5d4bf09dcf0b3a0f3f355336,front_door,Cote d'Ivoire,konan,60.75268981
14,west_african,5d4bdf23cf0b3a0f3f3372e1,front_door,Togo,tchamou,145.0
14,west_african,5d4bf35fcf0b3a0f3f359e80,front_door,Burkina Faso,sinare,44.71243293
14,west_african,5d4bf108cf0b3a0f3f355dc2,front_door,Cote d'Ivoire,fofana,41.76707666
14,west_african,5d4bdf31cf0b3a0f3f33744d,front_door,Togo,ayanou-ahobli,321.0
14,west_african,5d4bdf2ecf0b3a0f3f337413,front_door,Togo,kouleta,45.0
14,west_african,5d4bf3aacf0b3a0f3f35a6a2,front_door,Burkina Faso,zongo,55.93351031
14,west_african,5d4bdf2ecf0b3a0f3f337415,front_door,Togo,kouleta,45.0
14,west_african,5d4bdf28cf0b3a0f3f337369,front_door,Togo,messan,670.0
14,west_african,5d4bf460cf0b3a0f3f35b896,front_door,Burkina Faso,kabore,28.86121301
14,west_african,5d4bf0e5cf0b3a0f3f355ae4,front_door,Cote d'Ivoire,coulibaly,68.11250877
14,west_african,5d4bf460cf0b3a0f3f35b88c,front_door,Burkina Faso,kabore,28.86121301
14,west_african,5d4bf09dcf0b3a0f3f35533e,front_door,Cote d'Ivoire,konan,60.75268981
14,western,5ec4f912f0611d7ddd741436,front_door,Spain,sanchez,3195.0
14,western,5ec4f92df0611d7ddd741526,front_door,Spain,juan-caballero,7639.0
14,western,5d4be1e7cf0b3a0f3f33bcc3,front_door,United States,hoffman-tsuhlares,2347.0
14,western,5d4bde92cf0b3a0f3f33647d,front_door,Netherlands,rasing,3150.0
14,western,5ec4f920f0611d7ddd7414b0,front_door,Spain,rodrigo,4256.0
14,western,5ed42a29685870e51019a244,front_door,France,carrillo,7110.0
14,western,5d4be028cf0b3a0f3f338bb6,front_door,Netherlands,van-beek,3344.0
14,western,5f0259c41141893097043c53,front_door,Austria,posch,3003.0
14,western,5d4be0dacf0b3a0f3f339fe3,front_door,Netherlands,ploum,2370.0
14,western,5d4bed34cf0b3a0f3f34f2c0,front_door,United States,grabowski,2041.0
14,western,5d4bed34cf0b3a0f3f34f2c4,front_door,United States,grabowski,2041.0
14,western,5d4be035cf0b3a0f3f338d20,front_door,Netherlands,van-der-burgt,1816.0
15,west_african,5d4bf371cf0b3a0f3f35a07a,bike,Burkina Faso,mwe-quidraogo,53.64520422
15,west_african,5d4bdf27cf0b3a0f3f33735d,moped_motorcycle,Togo,messan,670.0
15,west_african,5d4bdf34cf0b3a0f3f3374a7,bike,Togo,ayanou-ahobli,321.0
15,west_african,5d4bf0c0cf0b3a0f3f355724,bike,Cote d'Ivoire,ahova,80.72770904
15,west_african,5d4bdf2fcf0b3a0f3f33742d,bike,Togo,kouleta,45.0
15,west_african,5d4bf3a8cf0b3a0f3f35a66e,bike,Burkina Faso,zongo,55.93351031
15,west_african,5d4bf38dcf0b3a0f3f35a3c0,bike,Burkina Faso,bande,52.95427615
15,west_african,5d4bdf2acf0b3a0f3f3373b3,bike,Togo,messan,670.0
15,west_african,5d4bf370cf0b3a0f3f35a070,moped_motorcycle,Burkina Faso,mwe-quidraogo,53.64520422
15,west_african,5d4bf3cacf0b3a0f3f35a964,bike,Burkina Faso,alimata,95.05494965
15,west_african,5d4bf052cf0b3a0f3f354bd0,bike,Liberia,galrbah,58.0
15,west_african,5d4be023cf0b3a0f3f338b48,bike,Nigeria,amama,3750.0
15,western,5d4bde8acf0b3a0f3f3363b7,car,Netherlands,rasing,3150.0
15,western,5ec4fbdff0611d7ddd742bc5,car,France,rayez,875.0
15,western,5ec4f54b3f62767d97a472e8,car,France,larriere,19671.0
15,western,5d4be028cf0b3a0f3f338bb8,car,Netherlands,van-beek,3344.0
15,western,5ec4f8f8f0611d7ddd741364,car,Spain,garrido,4253.0
15,western,5ec4f861f0611d7ddd740e42,car,United States,deveney,5125.0
15,western,5d4bde20cf0b3a0f3f3359fb,car,Italy,nazzani,4018.0
15,western,5ec4f5513f62767d97a4731f,car,France,larriere,19671.0
15,western,5ec4f8f2f0611d7ddd74132d,car,Spain,garrido,4253.0
15,western,5d4be3dacf0b3a0f3f33f0d3,car,United States,theresa-s,3450.0
15,western,5d4bde4ecf0b3a0f3f335e39,car,United States,stringini,960.0
15,western,5d4bde2bcf0b3a0f3f335af9,car,Italy,bravo,3536.0
16,west_african,5d4bdf2ccf0b3a0f3f3373d9,everyday_shoes,Togo,kouleta,45.0
16,west_african,5d4bdf26cf0b3a0f3f337341,everyday_shoes,Togo,messan,670.0
16,west_african,5d4bf0abcf0b3a0f3f3554c6,everyday_shoes,Cote d'Ivoire,yao-bernabi,82.91518231
16,west_african,5d4bf0f1cf0b3a0f3f355c56,everyday_shoes,Cote d'Ivoire,coulibaly,68.11250877
16,west_african,5d4bf469cf0b3a0f3f35b9a4,everyday_shoes,Burkina Faso,kabore,28.86121301
16,west_african,5d4bf08acf0b3a0f3f355178,everyday_shoes,Cote d'Ivoire,konan,60.75268981
16,west_african,5d4bf117cf0b3a0f3f355f5a,everyday_shoes,Cote d'Ivoire,fofana,41.76707666
16,west_african,5d4bdf21cf0b3a0f3f3372b3,everyday_shoes,Togo,tchamou,145.0
16,west_african,5d4bdf32cf0b3a0f3f337469,everyday_shoes,Togo,ayanou-ahobli,321.0
16,west_african,5d4bf0d3cf0b3a0f3f3558c8,everyday_shoes,Cote d'Ivoire,ahova,80.72770904
16,west_african,5d4bef1ecf0b3a0f3f352a74,everyday_shoes,Nigeria,ehegwo,59.0
16,west_african,5d4bf056cf0b3a0f3f354c02,everyday_shoes,Liberia,doe,87.0
16,west_african,5d4beefacf0b3a0f3f352640,everyday_shoes,Nigeria,nwadike,76.0
16,west_african,5d4bf04fcf0b3a0f3f354b8c,everyday_shoes,Liberia,galrbah,58.0
16,west_african,5ec4fb16f0611d7ddd74252f,everyday_shoes,Ghana,ampiah,213.0
16,west_african,5d4beee9cf0b3a0f3f352444,everyday_shoes,Nigeria,oputa-nwadioro,67.67970667
16,west_african,5d4bf049cf0b3a0f3f354af0,everyday_shoes,Liberia,nyanford,48.0
16,west_african,5d4be003cf0b3a0f3f338918,everyday_shoes,Nigeria,laoye,725.0
16,west_african,5d4bef07cf0b3a0f3f3527c0,everyday_shoes,Nigeria,udemba,41.0
16,west_african,5d4be013cf0b3a0f3f338a8c,everyday_shoes,Nigeria,amama,3750.0
16,west_african,5d4bef45cf0b3a0f3f352f12,everyday_shoes,Nigeria,shemede,124.2390709
16,west_african,5d4bf05bcf0b3a0f3f354c80,everyday_shoes,Liberia,blamoh,116.0
16,west_african,5ec4fad3f0611d7ddd7422fa,everyday_shoes,Ghana,nkrumah,375.0
16,west_african,5d4be087cf0b3a0f3f33965d,everyday_shoes,Nigeria,oke,277.0
16,western,5d4bde2ecf0b3a0f3f335b39,everyday_shoes,Italy,bravo,3536.0
16,western,5ec4f84bf0611d7ddd740d83,everyday_shoes,United States,deveney,5125.0
16,western,5d4be544cf0b3a0f3f3419cc,everyday_shoes,United States,macintyre,855.0
16,western,5d4be4ebcf0b3a0f3f340f86,everyday_shoes,United States,williams,2055.0
16,western,5ec4fbf1f0611d7ddd742c5f,everyday_shoes,France,rayez,875.0
16,western,5d4bde8ccf0b3a0f3f3363fd,everyday_shoes,Netherlands,rasing,3150.0
16,western,5d4be0dccf0b3a0f3f33a033,everyday_shoes,Netherlands,ploum,2370.0
16,western,5d4bdf60cf0b3a0f3f3378d1,everyday_shoes,Canada,vo-lowe,3360.0
16,western,5d4bdf9ecf0b3a0f3f337f0b,everyday_shoes,Switzerland,renato,2602.0
16,western,5d4be1edcf0b3a0f3f33bd77,everyday_shoes,United States,hoffman-tsuhlares,2347.0
16,western,5d4bde27cf0b3a0f3f335aa5,everyday_shoes,Italy,nazzani,4018.0
16,western,5d4be42dcf0b3a0f3f33fa62,everyday_shoes,United States,diamond,4446.0
17,west_african,5d4bdf30cf0b3a0f3f337433,clothes,Togo,kouleta,45.0
17,west_african,5d4bf08ecf0b3a0f3f3551d2,clothes,Cote d'Ivoire,konan,60.75268981
17,west_african,5d4bf3b9cf0b3a0f3f35a830,clothes,Burkina Faso,zongo,55.93351031
17,west_african,5d4bf369cf0b3a0f3f359fa0,clothes,Burkina Faso,sinare,44.71243293
17,west_african,5d4bf0afcf0b3a0f3f355544,clothes,Cote d'Ivoire,yao-bernabi,82.91518231
17,west_african,5d4bf39fcf0b3a0f3f35a56c,clothes,Burkina Faso,bande,52.95427615
17,west_african,5d4bf11acf0b3a0f3f355fbe,clothes,Cote d'Ivoire,fofana,41.76707666
17,west_african,5d4bdf25cf0b3a0f3f33730d,clothes,Togo,tchamou,145.0
17,west_african,5d4bf46ccf0b3a0f3f35b9f6,clothes,Burkina Faso,kabore,28.86121301
17,west_african,5d4bdf32cf0b3a0f3f33745b,clothes,Togo,ayanou-ahobli,321.0
17,west_african,5d4bf0d6cf0b3a0f3f355928,clothes,Cote d'Ivoire,ahova,80.72770904
17,west_african,5d4bf0e0cf0b3a0f3f355a5c,clothes,Cote d'Ivoire,coulibaly,68.11250877
17,west_african,5d4bf3dbcf0b3a0f3f35aa84,clothes,Burkina Faso,alimata,95.05494965
17,west_african,5d4bf37ecf0b3a0f3f35a208,clothes,Burkina Faso,mwe-quidraogo,53.64520422
17,west_african,5d4be00ecf0b3a0f3f338a12,clothes,Nigeria,amama,3750.0
17,west_african,5d4bf058cf0b3a0f3f354c40,clothes,Liberia,doe,87.0
17,west_african,5d4bef18cf0b3a0f3f3529c8,clothes,Nigeria,ibojionu,72.0
17,west_african,5d4be007cf0b3a0f3f338966,clothes,Nigeria,laoye,725.0
17,west_african,5d4bf051cf0b3a0f3f354bc4,clothes,Liberia,galrbah,58.0
17,west_african,5d4bef45cf0b3a0f3f352f00,clothes,Nigeria,shemede,124.2390709
17,west_african,5d4bf05ccf0b3a0f3f354ca4,clothes,Liberia,blamoh,116.0
17,west_african,5ec4fad6f0611d7ddd742314,clothes,Ghana,nkrumah,375.0
17,western,5d4be53fcf0b3a0f3f341932,clothes,United States,macintyre,855.0
17,western,5ec4fc55f0611d7ddd742f96,clothes,France,chamaitte,803.0
17,western,5d4bdf99cf0b3a0f3f337e81,clothes,Switzerland,renato,2602.0
17,western,5ed419e3685870535c19a206,clothes,France,carrillo,7110.0
17,western,5d4be537cf0b3a0f3f34185c,clothes,United States,diamond-01,2001.0
17,western,5d4bdf5fcf0b3a0f3f3378b3,clothes,Canada,vo-lowe,3360.0
17,western,5d4be4e6cf0b3a0f3f340ee8,clothes,United States,williams,2055.0
17,western,5d4be04acf0b3a0f3f338f4e,clothes,United States,janacek,4559.0
17,western,5ec4f8f7f0611d7ddd74135f,clothes,Spain,garrido,4253.0
17,western,5d4be050cf0b3a0f3f33901a,clothes,Netherlands,ploum,2370.0
17,western,5d4be1d2cf0b3a0f3f33babb,clothes,United States,hoffman-tsuhlares,2347.0
17,western,5d4be41ecf0b3a0f3f33f8a6,clothes,United States,diamond,4446.0
18,west_african,5d4bdf36cf0b3a0f3f3374bd,radio,Togo,ayanou-ahobli,321.0
18,west_african,5d4bf110cf0b3a0f3f355e92,phone,Cote d'Ivoire,fofana,41.76707666
18,west_african,5d4bf363cf0b3a0f3f359ef4,phone,Burkina Faso,sinare,44.71243293
18,west_african,5d4bf3d3cf0b3a0f3f35a9b8,phone,Burkina Faso,alimata,95.05494965
18,west_african,5d4bf0a7cf0b3a0f3f35544a,radio,Cote d'Ivoire,yao-bernabi,82.91518231
18,west_african,5d4bf084cf0b3a0f3f3550ca,phone,Cote d'Ivoire,konan,60.75268981
18,west_african,5d4bdf2ccf0b3a0f3f3373e1,radio,Togo,kouleta,45.0
18,west_african,5d4bf0ebcf0b3a0f3f355b94,phone,Cote d'Ivoire,coulibaly,68.11250877
18,west_african,5d4bdf27cf0b3a0f3f33734f,phone,Togo,messan,670.0
18,west_african,5d4bf0a3cf0b3a0f3f3553f0,phone,Cote d'Ivoire,yao-bernabi,82.91518231
18,west_african,5d4bf0c9cf0b3a0f3f35581e,phone,Cote d'Ivoire,ahova,80.72770904
18,west_african,5d4bdf32cf0b3a0f3f337457,phone,Togo,ayanou-ahobli,321.0
18,west_african,5d4bf376cf0b3a0f3f35a110,phone,Burkina Faso,mwe-quidraogo,53.64520422
18,west_african,5d4beef9cf0b3a0f3f35262c,radio,Nigeria,nwadike,76.0
18,west_african,5ec4fad2f0611d7ddd7422f3,phone,Ghana,nkrumah,375.0
18,west_african,5d4beee6cf0b3a0f3f3523fe,phone,Nigeria,oputa-nwadioro,67.67970667
18,west_african,5d4bef17cf0b3a0f3f35299a,radio,Nigeria,ibojionu,72.0
18,west_african,5d4bf04acf0b3a0f3f354af8,phone,Liberia,nyanford,48.0
18,west_african,5d4bef39cf0b3a0f3f352da8,phone,Nigeria,shemede,124.2390709
18,west_african,5d4bf05bcf0b3a0f3f354c86,radio,Liberia,blamoh,116.0
18,west_african,5d4beeedcf0b3a0f3f3524c0,radio,Nigeria,oputa-nwadioro,67.67970667
18,west_african,5d4be003cf0b3a0f3f338910,phone,Nigeria,laoye,725.0
18,west_african,5d4bf04acf0b3a0f3f354af4,radio,Liberia,nyanford,48.0
18,west_african,5d4be015cf0b3a0f3f338ac0,phone,Nigeria,amama,3750.0
18,western,5d4bde23cf0b3a0f3f335a47,tv,Italy,nazzani,4018.0
18,western,605a7a18d746332dd1aad5e9,tv,Denmark,hansen-refsing,5343.0
18,western,5ec4fc42f0611d7ddd742f02,tv,France,chamaitte,803.0
18,western,5d4bed2fcf0b3a0f3f34f23e,tv,United States,grabowski,2041.0
18,western,5d4be53ecf0b3a0f3f341930,tv,United States,macintyre,855.0
18,western,5ec4f90af0611d7ddd7413f0,tv,Spain,sanchez,3195.0
18,western,5d4be0d8cf0b3a0f3f339fbb,tv,Netherlands,ploum,2370.0
18,western,5d4be3cccf0b3a0f3f33ef2b,tv,United States,theresa-s,3450.0
18,western,5ec4f931f0611d7ddd741548,tv,Spain,tejada,2838.0
18,western,5d4be4e7cf0b3a0f3f340f1c,tv,United States,williams,2055.0
18,western,5ec4f8ecf0611d7ddd7412f9,tv,Spain,garrido,4253.0
18,western,5ec4f8faf0611d7ddd741375,tv,Spain,dela-torre-sawtana,4553.0
19,west_african,5d4bf39ecf0b3a0f3f35a54c,toothbrush,Burkina Faso,bande,52.95427615
19,west_african,5d4bf0f4cf0b3a0f3f355ca8,toothbrush,Cote d'Ivoire,coulibaly,68.11250877
19,west_african,5d4bf08dcf0b3a0f3f3551ae,toothbrush,Cote d'Ivoire,konan,60.75268981
19,west_african,5d4bdf2bcf0b3a0f3f3373c3,toothbrush,Togo,kouleta,45.0
19,west_african,5d4bf3dacf0b3a0f3f35aa72,toothbrush,Burkina Faso,alimata,95.05494965
19,west_african,5d4bf37dcf0b3a0f3f35a1ee,toothbrush,Burkina Faso,mwe-quidraogo,53.64520422
19,west_african,5d4bf369cf0b3a0f3f359f98,toothbrush,Burkina Faso,sinare,44.71243293
19,west_african,5d4bdf32cf0b3a0f3f337463,toothbrush,Togo,ayanou-ahobli,321.0
19,west_african,5d4bf11acf0b3a0f3f355fb2,toothbrush,Cote d'Ivoire,fofana,41.76707666
19,west_african,5d4bdf25cf0b3a0f3f337321,toothbrush,Togo,messan,670.0
19,west_african,5d4bf0afcf0b3a0f3f355532,toothbrush,Cote d'Ivoire,yao-bernabi,82.91518231
19,west_african,5d4bf0d4cf0b3a0f3f3558fa,toothbrush,Cote d'Ivoire,ahova,80.72770904
19,west_african,5d4bdf21cf0b3a0f3f3372a5,toothbrush,Togo,tchamou,145.0
19,west_african,5d4bf3c3cf0b3a0f3f35a8d8,toothbrush,Burkina Faso,zongo,55.93351031
19,west_african,5d4bef1fcf0b3a0f3f352a90,toothbrush,Nigeria,ehegwo,59.0
19,west_african,5d4bf055cf0b3a0f3f354be6,toothbrush,Liberia,doe,87.0
19,west_african,5d4beef8cf0b3a0f3f35260e,toothbrush,Nigeria,nwadike,76.0
19,west_african,5d4bf04dcf0b3a0f3f354b64,toothbrush,Liberia,galrbah,58.0
19,west_african,5ec4fb17f0611d7ddd742537,toothbrush,Ghana,ampiah,213.0
19,west_african,5d4beeebcf0b3a0f3f352482,toothbrush,Nigeria,oputa-nwadioro,67.67970667
19,west_african,5d4bf049cf0b3a0f3f354ad6,toothbrush,Liberia,nyanford,48.0
19,west_african,5d4be009cf0b3a0f3f33899c,toothbrush,Nigeria,laoye,725.0
19,west_african,5d4bef07cf0b3a0f3f3527c8,toothbrush,Nigeria,udemba,41.0
19,west_african,5d4be012cf0b3a0f3f338a62,toothbrush,Nigeria,amama,3750.0
19,western,5ec4fbf6f0611d7ddd742c86,toothbrush,France,rayez,875.0
19,western,5d4be542cf0b3a0f3f34198e,toothbrush,United States,macintyre,855.0
19,western,5d4be029cf0b3a0f3f338bd2,toothbrush,Netherlands,van-beek,3344.0
19,western,5d4be4eacf0b3a0f3f340f60,toothbrush,United States,williams,2055.0
19,western,5d4be1d4cf0b3a0f3f33bae9,toothbrush,United States,hoffman-tsuhlares,2347.0
19,western,605a7a18d746333cacaad5e6,toothbrush,Denmark,hansen-refsing,5343.0
19,western,5d4be403cf0b3a0f3f33f56c,toothbrush,United Kingdom,barnard,2010.0
19,western,5ec4f917f0611d7ddd741460,toothbrush,Spain,rodrigo,4256.0
19,western,5ec4f8ecf0611d7ddd7412fc,toothbrush,Spain,garrido,4253.0
19,western,5d4bde26cf0b3a0f3f335a8b,toothbrush,Italy,nazzani,4018.0
19,western,5ec4f5453f62767d97a472b8,toothbrush,France,larriere,19671.0
19,western,5ec4f82cf0611d7ddd740c80,toothbrush,United States,rivera,3478.0
20,west_african,5d4bdf35cf0b3a0f3f3374b1,chickens,Togo,ayanou-ahobli,321.0
20,west_african,5d4bf0b1cf0b3a0f3f35556a,goats,Cote d'Ivoire,yao-bernabi,82.91518231
20,west_african,5d4bf38fcf0b3a0f3f35a3e0,goats,Burkina Faso,bande,52.95427615
20,west_african,5d4bf0bacf0b3a0f3f355688,chickens,Cote d'Ivoire,yao-bernabi,82.91518231
20,west_african,5d4bf45bcf0b3a0f3f35b812,goats,Burkina Faso,kabore,28.86121301
20,west_african,5d4bf09fcf0b3a0f3f355372,chickens,Cote d'Ivoire,konan,60.75268981
20,west_african,5d4bf370cf0b3a0f3f35a066,chickens,Burkina Faso,mwe-quidraogo,53.64520422
20,west_african,5d4bf0dfcf0b3a0f3f355a32,chickens,Cote d'Ivoire,ahova,80.72770904
20,west_african,5d4bf04acf0b3a0f3f354b0e,chickens,Liberia,nyanford,48.0
20,west_african,5d4bef15cf0b3a0f3f352966,goats,Nigeria,ibojionu,72.0
20,west_african,5d4beef9cf0b3a0f3f352632,chickens,Nigeria,nwadike,76.0
20,west_african,5d4bef0bcf0b3a0f3f352830,goats,Nigeria,udemba,41.0
20,west_african,5d4bf051cf0b3a0f3f354bc8,chickens,Liberia,galrbah,58.0
20,western,5d4bde52cf0b3a0f3f335ea9,pet,United States,stringini,960.0
20,western,5ec4fbdef0611d7ddd742bb6,pet,France,rayez,875.0
20,western,5d4be027cf0b3a0f3f338bb0,pet,Netherlands,van-beek,3344.0
20,western,5d4be1dccf0b3a0f3f33bbab,pet,United States,hoffman-tsuhlares,2347.0
20,western,5d4be0d9cf0b3a0f3f339fd3,pet,Netherlands,ploum,2370.0
20,western,5ec4f84ff0611d7ddd740da4,pet,United States,deveney,5125.0
20,western,5d4be4e9cf0b3a0f3f340f42,pet,United States,williams,2055.0
20,western,5ec4f91af0611d7ddd74147f,pet,Spain,rodrigo,4256.0
20,western,5ec4f833f0611d7ddd740cbe,pet,United States,rivera,3478.0
20,western,5ec4f830f0611d7ddd740ca6,pet,United States,rivera,3478.0
20,western,6057da945ca47c7e11e75938,pet,Denmark,hansen-refsing,5343.0
20,western,5d4be3d2cf0b3a0f3f33efd1,pet,United States,theresa-s,3450.0
'''
cand = pd.read_csv(io.StringIO(CSV), dtype={'id': str})
print(len(cand), 'candidate photos')

In [ ]:
# index every image file of the input dataset by its Dollar Street id
paths = {}
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        stem, ext = os.path.splitext(f)
        if ext.lower() in ('.jpg', '.jpeg', '.png'):
            paths[stem] = os.path.join(root, f)
print(len(paths), 'image files found')
print(list(paths.items())[:2])
hit = cand['id'].isin(paths)
print('candidates found:', int(hit.sum()), 'of', len(cand))
print(cand.assign(found=hit).groupby(['row', 'group'])['found'].sum().unstack().T.to_string())

In [ ]:
os.makedirs('/kaggle/working/household', exist_ok=True)
rows = []
for r in cand[hit].itertuples():
    im = Image.open(paths[r.id]).convert('RGB')
    w0, h0 = im.size
    s = 448 / max(w0, h0)
    if s < 1:
        im = im.resize((round(w0 * s), round(h0 * s)), Image.LANCZOS)
    im.save(f'/kaggle/working/household/{r.id}.jpg', quality=92)
    rows.append({**r._asdict(), 'orig_w': w0, 'orig_h': h0, 'w': im.size[0], 'h': im.size[1]})
found = pd.DataFrame(rows).drop(columns=['Index'])
found.to_csv('/kaggle/working/household/found.csv', index=False)
with zipfile.ZipFile('/kaggle/working/household_candidates.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in os.listdir('/kaggle/working/household'):
        z.write(os.path.join('/kaggle/working/household', f), f)
print(len(found), 'photos, zip size MB', round(os.path.getsize('/kaggle/working/household_candidates.zip') / 1e6, 1))